# Explorando o Data Lakehouse

Este notebook é o laboratório exploratório da aula. Tudo já está conectado — você não precisa configurar host, porta, usuário ou senha de nada.

Uma coisa já existe pronta antes de você rodar qualquer célula: o pipeline **oficial** da lojinha — `customers`, `products`, `orders` e `order_items` do Postgres viram `lakehouse.bronze.*`, que viram `lakehouse.silver.sales`, que vira `lakehouse.gold.sales_by_day` / `sales_by_category`. Isso roda sozinho, uma vez, assim que o ambiente sobe (serviço `pipeline-init`, ver `trino/init/construir_pipeline.py`). Vamos usar um pedaço dele como apoio (a tabela `bronze.products`) e revisitar o resto lá no fim deste notebook.

O resto você constrói na mão, célula a célula, seguindo a arquitetura medalhão do começo ao fim — **landing -> bronze -> silver -> gold -> Postgres**. Só depois de fechar esse ciclo é que mostramos, como referência, as funções auxiliares de leitura (`lh.query` via Trino, `lh.postgres()` direto).

Enquanto as células rodam, dá pra acompanhar visualmente em duas telas:
- **MinIO Console** (http://localhost:9001) — vendo os arquivos aparecerem em `lakehouse/landing`, `/bronze`, `/silver`, `/gold`.
- **Trino UI** (http://localhost:8082) — vendo as queries `CREATE TABLE ... AS SELECT` rodando.

In [1]:
import lakehouse_kit as lh
import pandas as pd
import matplotlib.pyplot as plt

## 1. Landing — um arquivo "chega de fora"

Em aula real, nem todo dado começa num banco relacional: às vezes é um CSV que alguém te manda, um export de outro sistema. É pra isso que existe a pasta `landing` no bucket `lakehouse` — criada vazia (junto de bronze/silver/gold), esperando um arquivo chegar. Ninguém escreve nela sozinho, diferente das outras camadas.

Vamos simular isso: primeiro "chega" um CSV com o estoque atual de alguns produtos (aqui, geramos um na hora pra não depender de upload manual — em aula de verdade, seria um arquivo que você arrastou pro JupyterLab, que cairia aqui do lado do notebook, em `notebooks/`).

In [2]:
# Simula um CSV "chegando de fora"
pd.DataFrame({
    "produto": [
        "Fone de Ouvido Bluetooth",
        "Teclado Mecânico",
        "Mouse Sem Fio",
        "Cadeira de Escritório",
        "Garrafa Térmica",
    ],
    "estoque_atual": [42, 15, 60, 8, 120],
}).to_csv("estoque.csv", index=False)

lh.upload_to_landing("estoque.csv")
lh.list_landing()

['landing/.keep', 'landing/estoque.csv']

## 2. Landing -> Bronze — capture e cadastre no catálogo

Duas funções, dois passos:

- `lh.read_landing(arquivo)` **captura**: lê o arquivo direto da landing com pandas, sem passar pelo Trino — não importa se alguém já cadastrou tabela nenhuma, só se o arquivo existe no MinIO.
- `lh.write_table(df, layer, tabela)` **publica**: grava o DataFrame como Parquet numa camada do lakehouse (aqui, `bronze`) e cadastra a tabela no catálogo do Trino na mesma chamada. É o par que fecha o ciclo sem precisar escrever `CREATE TABLE ... WITH (external_location = ...)` na mão.

Depois de publicada, `lh.query(...)` — que sempre passa pelo Trino e só enxerga o que já foi cadastrado — já consegue ler.

In [3]:
# Captura: lê o arquivo direto da landing (sem passar pelo Trino)
estoque = lh.read_landing("estoque.csv")
estoque

,produto,estoque_atual
0,Fone de Ouvido Bluetooth,42
1,Teclado Mecânico,15
2,Mouse Sem Fio,60
3,Cadeira de Escritório,8
4,Garrafa Térmica,120


In [4]:
# Publica: já é uma tabela bronze de verdade, catalogada no Trino
lh.write_table(estoque, "bronze", "estoque")
lh.query("SELECT * FROM lakehouse.bronze.estoque")

,produto,estoque_atual
0,Fone de Ouvido Bluetooth,42
1,Teclado Mecânico,15
2,Mouse Sem Fio,60
3,Cadeira de Escritório,8
4,Garrafa Térmica,120


In [5]:
import os
os.remove("estoque.csv")  # só limpando o CSV local simulado acima — não é
                           # necessário se você arrastou um arquivo de verdade

## 3. Bronze -> Silver — enriquecendo com uma dimensão

Sozinha, `bronze.estoque` não diz muito: só nome do produto e quantidade. A camada silver junta esse dado com uma dimensão — aqui, `lakehouse.bronze.products` (que já veio pronta do pipeline oficial: também é só uma camada bronze, construída da mesma forma, só que a partir do Postgres em vez da landing) — pra saber categoria, preço, e quanto isso representa em reais parado em estoque.

Assim como o pipeline oficial faz para `silver.sales`, a silver aqui é construída direto em SQL com `lh.run_sql(...)` (um `CREATE TABLE ... AS SELECT`), em vez de `write_table` — as duas formas chegam no mesmo lugar: uma tabela cadastrada no catálogo, com Parquet no MinIO por baixo.

In [6]:
lh.run_sql(f"CREATE SCHEMA IF NOT EXISTS lakehouse.silver WITH (location = 's3://{lh.bucket()}/silver/')")
lh.run_sql("DROP TABLE IF EXISTS lakehouse.silver.estoque_valorizado")
lh.run_sql(
    """
    CREATE TABLE lakehouse.silver.estoque_valorizado
    WITH (format = 'PARQUET')
    AS SELECT
        e.produto,
        p.category AS categoria,
        p.price    AS preco_unitario,
        e.estoque_atual,
        e.estoque_atual * p.price AS valor_em_estoque
    FROM lakehouse.bronze.estoque e
    JOIN lakehouse.bronze.products p ON e.produto = p.name
    """
)
lh.query("SELECT * FROM lakehouse.silver.estoque_valorizado")

,produto,categoria,preco_unitario,estoque_atual,valor_em_estoque
0,Fone de Ouvido Bluetooth,Eletrônicos,129.9,42,5455.8
1,Teclado Mecânico,Eletrônicos,249.9,15,3748.5
2,Mouse Sem Fio,Eletrônicos,79.9,60,4794.0
3,Cadeira de Escritório,Móveis,649.0,8,5192.0
4,Garrafa Térmica,Casa,59.9,120,7188.0


## 4. Silver -> Gold — agregado pronto para consumo

Gold é a camada que alguém de negócio consumiria direto: pouca linha, já resumida, sem precisar entender join nenhum. Aqui, quanto de estoque (em unidades e em R$) cada categoria representa.

In [ ]:
lh.run_sql(f"CREATE SCHEMA IF NOT EXISTS lakehouse.gold WITH (location = 's3://{lh.bucket()}/gold/')")
lh.run_sql("DROP TABLE IF EXISTS lakehouse.gold.estoque_por_categoria")
lh.run_sql(
    """
    CREATE TABLE lakehouse.gold.estoque_por_categoria
    WITH (format = 'PARQUET')
    AS SELECT
        categoria,
        SUM(estoque_atual)    AS unidades_em_estoque,
        SUM(valor_em_estoque) AS valor_total
    FROM lakehouse.silver.estoque_valorizado
    GROUP BY categoria
    ORDER BY valor_total DESC
    """
)
gold_estoque = lh.query("SELECT * FROM lakehouse.gold.estoque_por_categoria")
gold_estoque

In [ ]:
gold_estoque.plot(x="categoria", y="valor_total", kind="bar", figsize=(8, 4), title="Valor em estoque por categoria", legend=False)
plt.tight_layout()
plt.show()

## 5. Gold -> Postgres — devolvendo o resultado pro sistema operacional

O lakehouse não é o fim da linha: às vezes o consumidor final (um dashboard, um sistema interno, alguém que só sabe abrir o Postgres) não fala Trino nem Parquet — só SQL sobre um banco relacional comum. Esse é o passo de "reverse ETL": pegar um resultado já pronto do gold e gravar de volta no Postgres, como uma tabela normal.

`lh.postgres()` devolve uma engine SQLAlchemy — dá pra usar `DataFrame.to_sql(...)` do pandas normalmente, sem nada de especial do lakehouse envolvido.

In [ ]:
pg = lh.postgres()
gold_estoque.to_sql("estoque_por_categoria", pg, if_exists="replace", index=False)

# Conferindo: lendo de volta direto do Postgres, sem passar pelo lakehouse
pd.read_sql("SELECT * FROM estoque_por_categoria", pg)

In [ ]:
# Descomente para desfazer tudo que este notebook construiu
# (landing, bronze, silver, gold e a tabela no Postgres):

# from sqlalchemy import text
# lh.drop_table("gold", "estoque_por_categoria")
# lh.drop_table("silver", "estoque_valorizado")
# lh.drop_table("bronze", "estoque")
# lh.s3().delete_object(Bucket=lh.bucket(), Key="landing/estoque.csv")
# with lh.postgres().begin() as conn:
#     conn.execute(text("DROP TABLE IF EXISTS estoque_por_categoria"))

## 6. Funções auxiliares de leitura (Trino e Postgres)

Fechado o ciclo landing -> bronze -> silver -> gold -> Postgres, aqui vai — como referência, tudo num lugar só — o conjunto de funções que servem só para **ler** dado, dos dois lados do lakehouse:

**Do lado do Trino / MinIO:**
- `lh.list_layer(layer)` lista os arquivos físicos de uma camada (sem ler o conteúdo).
- `lh.query(sql)` executa SQL no Trino e devolve um DataFrame — só enxerga o que já foi **cadastrado** no catálogo (`lakehouse.<layer>.<tabela>`).
- `lh.read_table(layer, tabela)` lê o Parquet direto do MinIO com pandas — não importa se a tabela foi cadastrada no Trino ou não, só se o arquivo existe.

**Do lado do Postgres:**
- `lh.postgres()` devolve uma engine SQLAlchemy pro Postgres "fonte" (o mesmo pra onde você escreveu `estoque_por_categoria` no passo 5) — dá pra usar com `pd.read_sql(...)` ou `pd.read_sql_table(...)`, direto, sem passar pelo lakehouse.

Vamos usar essas funções para dar uma olhada no que o pipeline **oficial** já tinha construído antes deste notebook começar — e que a gente ainda não tinha visto até agora.

In [ ]:
# O que existe fisicamente na camada bronze — inclui a "estoque" que a gente
# publicou no passo 2, e as 4 tabelas do pipeline oficial (customers, products,
# orders, order_items)
lh.list_layer("bronze")

In [ ]:
lh.query("SHOW SCHEMAS FROM lakehouse")

In [ ]:
# Lendo o mesmo bronze.estoque, mas agora direto do MinIO via pandas,
# sem passar pelo Trino — pra comparar com lh.query() do passo 2
lh.read_table("bronze", "estoque")

In [ ]:
# lakehouse.silver.sales e lakehouse.gold.* já existiam antes deste notebook:
# construídos pelo pipeline oficial (customers/products/orders/order_items ->
# silver.sales -> gold.sales_by_day / sales_by_category), que roda sozinho no
# serviço pipeline-init assim que o ambiente sobe.
gold_por_dia = lh.query("SELECT * FROM lakehouse.gold.sales_by_day ORDER BY order_date")
gold_por_dia.plot(x="order_date", y="receita", kind="line", marker="o", figsize=(10, 4), title="Receita por dia")
plt.tight_layout()
plt.show()

In [ ]:
gold_por_categoria = lh.query("SELECT * FROM lakehouse.gold.sales_by_category ORDER BY receita DESC")
gold_por_categoria.plot(x="product_category", y="receita", kind="bar", figsize=(8, 4), title="Receita por categoria", legend=False)
plt.tight_layout()
plt.show()

In [ ]:
# Lendo direto do Postgres, sem passar pelo lakehouse — a fonte original...
pd.read_sql("SELECT * FROM customers LIMIT 5", lh.postgres())

In [ ]:
# ...e a tabela que a gente mesmo escreveu de volta no passo 5
pd.read_sql("SELECT * FROM estoque_por_categoria", lh.postgres())

## 7. Bônus — construa sua própria camada gold

Tudo que você fez nas seções 2 a 5 foi com `lh.write_table(...)` e `lh.run_sql(...)` — não tem nada de especial por trás. Exemplo, reaproveitando a `silver.sales` oficial: um "top 3 clientes por receita" que não existe no pipeline oficial.

In [ ]:
top_clientes = lh.query("SELECT customer_name, SUM(item_total) AS receita FROM lakehouse.silver.sales GROUP BY customer_name")
top_clientes = top_clientes.sort_values("receita", ascending=False).head(3).reset_index(drop=True)

lh.write_table(top_clientes, "gold", "top_clientes")
lh.query("SELECT * FROM lakehouse.gold.top_clientes")

In [ ]:
# lh.drop_table("gold", "top_clientes")  # descomente pra desfazer (tira do catálogo E apaga o Parquet)

## 8. Para explorar em aula

- Insira um novo pedido direto no Postgres (`lh.postgres()` + um `INSERT`) e rode o pipeline oficial de novo pra atualizar o lakehouse — de dentro deste notebook, numa célula nova: `!python /home/jovyan/trino-init/construir_pipeline.py`. Depois, rode de novo as células da seção 6 e veja o `gold.sales_by_day` mudar.
- Abra o MinIO Console e compare o tamanho/quantidade de arquivos entre bronze e gold — por que gold tem menos dado?
- Escreva uma nova query de agregação (ex: receita por cliente) direto em `lh.query(...)`.
- Arraste um arquivo de verdade (CSV, JSON ou Parquet) pro JupyterLab e repita o fluxo das seções 1-2 (`upload_to_landing` -> `read_landing` -> `write_table`) com ele, em vez do CSV simulado.
- Escreva sua própria camada silver/gold a partir de outra pergunta de negócio, e devolva o resultado pro Postgres como no passo 5.
- Derrube tudo com `docker compose down -v` e suba de novo — o ambiente inteiro volta ao estado zero (e o pipeline oficial roda sozinho de novo).